# 9.7 — Output Modes

**Chapter 9, §9.7.2, and Exercise 6.** *Why does Spark refuse to append the result
of an aggregation, and what changes when you add a watermark?*

An output mode is a promise about the rows that reach the sink. Append promises
that a row, once written, will never need revising. Update promises only that what
it writes has changed since last time. Complete promises the whole result table,
every trigger.

Spark will not let a query make a promise it cannot keep, so the choice of mode is
not a preference. This notebook provokes the refusal, reads the error, then earns
append back by adding the one thing that makes finality decidable.

In [1]:
import logging, os, shutil, tempfile, time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window
from pyspark.errors import AnalysisException

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch09-output-modes")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

spark = (
    SparkSession.builder.appName("CS777-9.7")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# PySpark logs a structured record, JVM stack trace and all, for every analysis
# error. This notebook raises one on purpose, so quiet that logger and keep the
# message readable.
logging.getLogger("DataFrameQueryContextLogger").setLevel(logging.CRITICAL)
spark.version

'4.2.0'

In [2]:
DAY = "2026-01-01"
BATCHES = [
    [("12:02", "dog"), ("12:04", "cat"), ("12:07", "dog"), ("12:09", "cat"), ("12:14", "dog")],
    [("12:21", "owl"), ("12:23", "owl")],
    [("12:31", "cat"), ("12:34", "dog")],
    [("12:41", "owl")],
]
INPUT = os.path.join(SCRATCH, "input")
os.makedirs(INPUT, exist_ok=True)
for order, rows in enumerate(BATCHES):
    path = os.path.join(INPUT, f"{order}.csv")
    with open(path, "w") as fh:
        for clock, word in rows:
            fh.write(f"{DAY} {clock}:00,{word}\n")
    stamp = time.time() - 3600 + 10 * order
    os.utime(path, (stamp, stamp))

SCHEMA = "event_time TIMESTAMP, word STRING"

def stream():
    return (
        spark.readStream.schema(SCHEMA)
        .option("maxFilesPerTrigger", 1)
        .csv(INPUT)
    )

def run(frame, name, mode):
    query = (
        frame.writeStream.queryName(name)
        .outputMode(mode)
        .format("memory")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
    return query

print(f"{len(BATCHES)} files, {sum(len(b) for b in BATCHES)} rows")

4 files, 10 rows


## Append, refused

A running count over a stream with no watermark can change at any time, because a
record for an old group may always turn up. No row is ever provably final, so
append has no legal output and the query is rejected before it starts.

In [3]:
running_count = stream().groupBy("word").count()

try:
    run(running_count, "refused", "append")
    print("no error: unexpected")
except AnalysisException as exc:
    print(str(exc).strip().split("\n")[0])

[STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: append. This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets. SQLSTATE: 42KDE


## Update and complete, accepted

The same query is fine in either of the other two modes, because neither claims
finality. Update emits the groups whose counts changed in this trigger; complete
rewrites the entire result table every trigger and therefore never discards any
of its state.

In [4]:
run(stream().groupBy("word").count(), "in_update", "update")
print("update mode, rows emitted across all triggers:", spark.table("in_update").count())

run(stream().groupBy("word").count(), "in_complete", "complete")
spark.table("in_complete").orderBy("word").show(truncate=False)

update mode, rows emitted across all triggers: 6


+----+-----+
|word|count|
+----+-----+
|cat |3    |
|dog |4    |
|owl |3    |
+----+-----+



## Append, earned back

Add an event-time window and a watermark and finality becomes decidable: once the
watermark passes the end of a window, that window's count can no longer change and
the row is safe to append.

The cost is latency. Nothing is emitted while a window is still open, so every
result arrives at least one watermark behind the data that produced it. The
windows below that have not yet been passed by the watermark are simply absent
from the output, not wrong.

In [5]:
windowed = (
    stream()
    .withWatermark("event_time", "5 minutes")
    .groupBy(window(col("event_time"), "10 minutes"), col("word"))
    .count()
)
query = run(windowed, "in_append", "append")

for progress in query.recentProgress:
    print(
        f"batch {progress['batchId']}: read {progress['numInputRows']}, "
        f"emitted {progress['sink']['numOutputRows']}, "
        f"watermark reported {progress['eventTime'].get('watermark', '-')[11:16]}"
    )

batch 0: read 5, emitted 0, watermark reported 00:00
batch 1: read 2, emitted 0, watermark reported 12:09
batch 2: read 2, emitted 2, watermark reported 12:18
batch 3: read 1, emitted 1, watermark reported 12:29
batch 4: read 0, emitted 1, watermark reported 12:36


In [6]:
# Only the windows the watermark has finished with. The last two, 12:30-12:40 and
# 12:40-12:50, are still open, so they are absent rather than wrong.
spark.table("in_append").orderBy("window", "word").show(truncate=False)

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|cat |2    |
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|dog |2    |
|{2026-01-01 12:10:00, 2026-01-01 12:20:00}|dog |1    |
|{2026-01-01 12:20:00, 2026-01-01 12:30:00}|owl |2    |
+------------------------------------------+----+-----+



## The rule in one table

Reading it downward, the single principle is that a mode is allowed only when its
promise about emitted rows can be kept.

In [7]:
def allowed(frame, mode, name):
    try:
        run(frame(), name, mode)
        return "yes"
    except AnalysisException:
        return "no"

rows = []
for label, frame in [
    ("aggregation on event time, with watermark",
     lambda: stream().withWatermark("event_time", "5 minutes")
                     .groupBy(window(col("event_time"), "10 minutes")).count()),
    ("aggregation without a watermark",
     lambda: stream().groupBy("word").count()),
    ("no aggregation",
     lambda: stream().where(col("word") != "owl")),
]:
    verdicts = {m: allowed(frame, m, f"probe_{len(rows)}_{m}")
                for m in ("append", "update", "complete")}
    rows.append((label, verdicts))

print(f"{'query type':<44} {'append':>8} {'update':>8} {'complete':>10}")
for label, verdicts in rows:
    print(f"{label:<44} {verdicts['append']:>8} {verdicts['update']:>8} {verdicts['complete']:>10}")

query type                                     append   update   complete
aggregation on event time, with watermark         yes      yes        yes
aggregation without a watermark                    no      yes        yes
no aggregation                                    yes      yes         no


## What this shows

The compatibility table in the chapter is not a list to memorize; it falls out of
one question asked three times. Can this row still change? If it can never change,
append is legal. If it can, append is not, and the choice is between update, which
publishes revisions as they happen, and complete, which republishes everything and
therefore holds every group's state forever.

Update is usually the right default. For a query with no aggregation it behaves
exactly like append, and it supports far more queries than either alternative. The
usual production shape is update mode into a store that accepts upserts, with the
full table assembled downstream when someone actually needs it.

In [8]:
spark.stop()